
## Organismos selecionados

Os organismos foram selecionados a partir de:

- **Millán Arias et al. (2023)**: Supplementary Table S1, que reúne ~700 genomas procarióticos com temperatura ótima de crescimento (OGT) conhecida. Quatro dos sete organismos foram retirados diretamente dessa tabela. (os mesmos do trabalho 1)
- **Literatura de extremófilos**: Três organismos foram adicionados além da tabela original para garantir diversidade filogenética entre as bactérias hipertermófilas e termófilas.

| Organismo | Domínio | Filo | Categoria | OGT | Assembly NCBI | Fonte |
|---|---|---|---|---|---|---|
| *Thermocrinis ruber* | Bacteria | Aquificae | Hipertermófilo | >80°C | GCA_000512735.1 | Millán Arias |
| *Thermotoga maritima* | Bacteria | Thermotogae | Hipertermófilo | >80°C | GCA_000008545.1 | Millán Arias |
| *Aquifex aeolicus* | Bacteria | Aquificae | Hipertermófilo | ~95°C | GCA_000008625.1 | Literatura |
| *Thermus thermophilus* | Bacteria | Deinococcus-Thermus | Termófilo | ~65°C | GCA_000196015.1 | Literatura |
| *Escherichia fergusonii* | Bacteria | Proteobacteria | Mesófilo | 20–45°C | GCA_000026225.1 | Millán Arias |
| *Psychrobacter arcticus* | Bacteria | Proteobacteria | Psicrófilo | <20°C | GCA_000012305.1 | Millán Arias |
| *Psychromonas ingrahamii* | Bacteria | Proteobacteria | Psicrófilo | −12°C | GCA_000015285.1 | Millán Arias |

---

## Genes analisados

Os quatro genes foram selecionados no Trabalho 1 com base em Verma et al. (2024) por seu envolvimento direto em respostas ao estresse térmico.

| Gene | KO Number | Função |
|---|---|---|
| dnaK | K04043 | Chaperona Hsp70 — dobramento e estabilização de proteínas |
| groEL | K04077 | Chaperonina Hsp60 — remontagem de proteínas desnaturadas |
| gyrA | K02469 | DNA girase subunidade A — regulação do superenrolamento do DNA |
| deaD | K05592 | RNA helicase DEAD-box — resolução de estruturas de RNA no frio |

---

## Presença dos genes por organismo

| Organismo | dnaK | groEL | gyrA | deaD |
|---|---|---|---|---|
| *T. ruber* | ✓ | ✓ | ✓ | ✓ |
| *T. maritima* | ✓ | ✓ | ✓ | ✗ |
| *A. aeolicus* | ✓ | ✓ | ✓ | ** |
| *T. thermophilus* | ✓ | ✓ | ✓ | ✓ |
| *E. fergusonii* | ✓ | ✓ | ✓ | ✓ |
| *P. arcticus* | ✓ | ✓ | ✓ | ✗ |
| *P. ingrahamii* | ✓ | ✓ | ✓ | ✓ |



> ** O gene deaD de *A. aeolicus* (aq_613) está anotado funcionalmente no KEGG mas sem KO number atribuído (K05592). Foi incluído na análise com base na anotação funcional e nomenclatura do gene.
>
> Ausência confirmada no mapeamento KEGG - organismos excluídos do alinhamento de deaD.




In [ ]:
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
import requests
import os
import time
import subprocess


## 1. Obtenção das sequências nucleotídicas via KEGG

Após a identificação dos genes, as sequências nucleotídicas codificantes (CDS) foram obtidas diretamente por meio da API REST do KEGG. Para cada gene foi realizada uma requisição utilizando o identificador KEGG correspondente:

```text
https://rest.kegg.jp/get/<gene_id>/ntseq
```

Por exemplo, para o gene *deaD* de *Thermocrinis ruber*:

```text
https://rest.kegg.jp/get/trd:THERU_06610/ntseq
```

Esse endpoint retorna diretamente a sequência nucleotídica codificante associada ao gene

In [ ]:

# IDs KEGG dos genes por organismo
genes_kegg = {
    "T_ruber": {
        "dnaK":  "trd:THERU_01435",
        "groEL": "trd:THERU_06905",
        "gyrA":  "trd:THERU_01910",
        "deaD":  "trd:THERU_06610",
    },
    "T_maritima": {
        "dnaK":  "tma:TM0373",
        "groEL": "tma:TM0506",
        "gyrA":  "tma:TM1084",
        "deaD":  None,           
    },
    "A_aeolicus": {
        "dnaK":  "aae:aq_996",
        "groEL": "aae:aq_2200",
        "gyrA":  "aae:aq_980",
        "deaD":  "aae:aq_613",   
    },
    "T_thermophilus": {
        "dnaK":  "tth:TT_C1127",
        "groEL": "tth:TT_C1714", 
        "gyrA":  "tth:TT_C0990",
        "deaD":  "tth:TT_C1895",
    },
    "E_fergusonii": {
        "dnaK":  "efe:EFER_0010",
        "groEL": "efe:EFER_4195",
        "gyrA":  "efe:EFER_0934",
        "deaD":  "efe:EFER_3141",
    },
    "P_arcticus": {
        "dnaK":  "par:Psyc_2132",
        "groEL": "par:Psyc_0553",
        "gyrA":  "par:Psyc_1543",
        "deaD":  None,           
    },
    "P_ingrahamii": {
        "dnaK":  "pin:Ping_0917",
        "groEL": "pin:Ping_0844",
        "gyrA":  "pin:Ping_1114",
        "deaD":  "pin:Ping_3203",
    },
}

In [ ]:
def baixar_ntseq(kegg_id):
    """Baixa a sequência nucleotídica de um gene via API REST do KEGG."""
    url = f"https://rest.kegg.jp/get/{kegg_id}/ntseq"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Erro HTTP {r.status_code} para {kegg_id}")
    texto = r.text.strip()
    if not texto.startswith(">"):
        raise RuntimeError(f"Resposta inesperada para {kegg_id}")
    return texto


os.makedirs("sequencias_nt", exist_ok=True)

por_gene = {"dnaK": [], "groEL": [], "gyrA": [], "deaD": []}
falhas = []

print("=" * 60)
print("Baixando sequências nucleotídicas via KEGG REST API")
print("=" * 60)

for organismo, genes in genes_kegg.items():
    print(f"\n── {organismo}")
    for gene, kegg_id in genes.items():
        if kegg_id is None:
            print(f"  {gene}: ausente")
            continue
        print(f"  {gene} ({kegg_id})... ", end="", flush=True)
        try:
            fasta = baixar_ntseq(kegg_id)
            linhas = fasta.splitlines()
            sequencia = "".join(l.strip() for l in linhas if not l.startswith(">"))
            fasta_final = f">{organismo}_{gene}\n{sequencia}"
            por_gene[gene].append(fasta_final)
            print("ok")
        except Exception as e:
            print(f"{e}")
            falhas.append((organismo, gene, str(e)))
        time.sleep(0.5)

print("\n" + "=" * 60)
print("Salvando arquivos FASTA")
print("=" * 60)

for gene, seqs in por_gene.items():
    if not seqs:
        continue
    arquivo = f"sequencias_nt/{gene}_sequencias.fasta"
    with open(arquivo, "w") as f:
        f.write("\n\n".join(seqs))
    print(f"  {gene}: {len(seqs)} sequências → {arquivo}")

if falhas:
    print("\nFalhas:")
    for org, gene, erro in falhas:
        print(f"  {org} / {gene}: {erro}")

Baixando sequências nucleotídicas via KEGG REST API

── T_ruber
  dnaK (trd:THERU_01435)... ✓
  groEL (trd:THERU_06905)... ✓
  gyrA (trd:THERU_01910)... ✓
  deaD (trd:THERU_06610)... ✓

── T_maritima
  dnaK (tma:TM0373)... ✓
  groEL (tma:TM0506)... ✓
  gyrA (tma:TM1084)... ✓
  deaD: ausente

── A_aeolicus
  dnaK (aae:aq_996)... ✓
  groEL (aae:aq_2200)... ✓
  gyrA (aae:aq_980)... ✓
  deaD (aae:aq_613)... ✓

── T_thermophilus
  dnaK (tth:TT_C1127)... ✓
  groEL (tth:TT_C1714)... ✓
  gyrA (tth:TT_C0990)... ✓
  deaD (tth:TT_C1895)... ✓

── E_fergusonii
  dnaK (efe:EFER_0010)... ✓
  groEL (efe:EFER_4195)... ✓
  gyrA (efe:EFER_0934)... ✓
  deaD (efe:EFER_3141)... ✓

── P_arcticus
  dnaK (par:Psyc_2132)... ✓
  groEL (par:Psyc_0553)... ✓
  gyrA (par:Psyc_1543)... ✓
  deaD: ausente

── P_ingrahamii
  dnaK (pin:Ping_0917)... ✓
  groEL (pin:Ping_0844)... ✓
  gyrA (pin:Ping_1114)... ✓
  deaD (pin:Ping_3203)... ✓

Salvando arquivos FASTA
  dnaK: 7 sequências → sequencias_nt/dnaK_sequencias.fasta
  g

## 2. Alinhamento múltiplo com MAFFT

Cada gene é alinhado separadamente com MAFFT usando a opção `--auto`, que seleciona automaticamente o algoritmo mais adequado para o conjunto de sequências 

O deaD é alinhado com 5 organismos (T. maritima e P. arcticus ausentes).

In [ ]:
os.makedirs("alinhamentos", exist_ok=True)

genes = ["dnaK", "groEL", "gyrA", "deaD"]

print("Rodando MAFFT...\n")

for gene in genes:
    entrada = f"sequencias_nt/{gene}_sequencias.fasta"
    saida   = f"alinhamentos/{gene}_alinhado.fasta"

    if not os.path.exists(entrada):
        print(f"  {gene}: arquivo não encontrado")
        continue

    resultado = subprocess.run(
        ["mafft", "--auto", entrada],
        capture_output=True,
        text=True
    )

    if resultado.returncode != 0:
        print(f"  {gene}: Erro\n{resultado.stderr}")
        continue

    with open(saida, "w") as f:
        f.write(resultado.stdout)

    # Extrair informações relevantes do log
    linhas = resultado.stderr.split("\n")

    algoritmo = next((l.strip() for l in linhas if "L-INS-i" in l or "FFT-NS" in l or "G-INS-i" in l), "—")
    modelo    = next((l.strip() for l in linhas if "model=" in l and "alg=L" in l), "")
    modelo    = modelo.split("model=")[1].split(",")[0].strip() if "model=" in modelo else "—"
    gap       = next((l.strip() for l in linhas if "Gap Penalty" in l), "—")
    n_seqs    = resultado.stdout.count(">")

    print(f"  {gene}")
    print(f"    Sequências:  {n_seqs}")
    print(f"    Algoritmo:   {algoritmo}")
    print(f"    Modelo:      {modelo}")
    print(f"    {gap}")
    print(f"    Saída:       {saida}")
    print()

print("Concluído.")

Rodando MAFFT...

  dnaK
    Sequências:  7
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/dnaK_alinhado.fasta

  groEL
    Sequências:  7
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/groEL_alinhado.fasta

  gyrA
    Sequências:  7
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/gyrA_alinhado.fasta

  deaD
    Sequências:  5
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/deaD_alinhado.fasta

Concluído.


## 3. Verificação dos alinhamentos

Conferir número de sequências e comprimento do alinhamento para cada gene.

In [ ]:

print(f"{'Gene':<8} {'Sequências':>12} {'Comprimento (bp)':>18}")
print("-" * 42)

for gene in genes:
    arquivo = f"alinhamentos/{gene}_alinhado.fasta"
    if not os.path.exists(arquivo):
        print(f"{gene:<8} {'—':>12} {'—':>18}")
        continue
    aln = AlignIO.read(arquivo, "fasta")
    print(f"{gene:<8} {len(aln):>12} {aln.get_alignment_length():>18}")



Gene       Sequências   Comprimento (bp)
------------------------------------------
dnaK                7               1992
groEL               7               1662
gyrA                7               2913
deaD                5               1988


## 4. Concatenação para a árvore MLSA

Os alinhamentos de dnaK, groEL e gyrA são concatenados em uma sequência única por organismo.
O deaD é excluído da concatenação por estar ausente em T. maritima e P. arcticus.

Metodologia baseada em Bouvet et al. (2014), que usaram o software START2 para concatenar 6 genes funcionais antes de construir a árvore MLSA.

In [21]:
genes_mlsa = ["dnaK", "groEL", "gyrA"]

# Carregar alinhamentos removendo sufixo do gene do ID
alinhamentos = {}
for gene in genes_mlsa:
    aln = AlignIO.read(f"alinhamentos/{gene}_alinhado.fasta", "fasta")
    alinhamentos[gene] = {
        rec.id.replace(f"_{gene}", ""): str(rec.seq)
        for rec in aln
    }

# Organismos presentes em todos os genes
organismos = set(alinhamentos[genes_mlsa[0]].keys())
for gene in genes_mlsa[1:]:
    organismos &= set(alinhamentos[gene].keys())

# Concatenar
concatenadas = []
for org in sorted(organismos):
    seq_concat = "".join(alinhamentos[gene][org] for gene in genes_mlsa)
    concatenadas.append(SeqRecord(Seq(seq_concat), id=org, description=""))

SeqIO.write(concatenadas, "alinhamentos/concatenado_mlsa.fasta", "fasta")

# Resumo
comprimentos = {gene: aln.get_alignment_length()
                for gene, aln in
                [(g, AlignIO.read(f"alinhamentos/{g}_alinhado.fasta", "fasta"))
                 for g in genes_mlsa]}
total = sum(comprimentos.values())

print("Alinhamentos individuais:")
print(f"  dnaK:  {comprimentos['dnaK']} bp")
print(f"  groEL: {comprimentos['groEL']} bp")
print(f"  gyrA:  {comprimentos['gyrA']} bp")
print(f"  Total: {comprimentos['dnaK']} + {comprimentos['groEL']} + {comprimentos['gyrA']} = {total} bp")
print(f"\nOrganismos na concatenada: {len(concatenadas)}")
for rec in concatenadas:
    print(f"  {rec.id}  ({len(rec.seq)} bp)")
print(f"\nSalvo: alinhamentos/concatenado_mlsa.fasta")

Alinhamentos individuais:
  dnaK:  1992 bp
  groEL: 1662 bp
  gyrA:  2913 bp
  Total: 1992 + 1662 + 2913 = 6567 bp

Organismos na concatenada: 7
  A_aeolicus  (6567 bp)
  E_fergusonii  (6567 bp)
  P_arcticus  (6567 bp)
  P_ingrahamii  (6567 bp)
  T_maritima  (6567 bp)
  T_ruber  (6567 bp)
  T_thermophilus  (6567 bp)

Salvo: alinhamentos/concatenado_mlsa.fasta
